<a href="https://colab.research.google.com/github/jiuwong/sfu_AppliedAI_DataAnalytics/blob/main/6_1_hands_on_with_ml_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://sfudial.ca/wp-content/uploads/SFU-DIAL-Logo.png" width=40%>&nbsp;&nbsp;&nbsp;&nbsp;<img src="https://www.sfu.ca/content/dam/sfu/images/brand_extension/SFU-Big-Data_Logo.png" width=40%>

# Lab 6.1: Deploying an AI Model on Google Sheets with MLOps Principles

Master AI-enhanced MLOps workflows for real-world model deployment. Learn to deploy models to Google Sheets, monitor performance, and manage the complete ML lifecycle while developing critical thinking skills for production deployment decisions.

**Use the TODOs and prompt your AI like a teammate. Think critically, experiment often, and document your process.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/gist/git-steb/c5f1f44427d5b78bd90ef84dd237bb2f/6_1_Hands_On_with_ML_Ops.ipynb)

**Note:** For ease of following and to save training time on large datasets, this lab focuses on deploying data analysis insights onto Google Sheets. If you find this easy to follow, consider trying to deploy an actual ML prediction model on this dataset as a next step. We will provide some prompts you can try at the end.

## Lab Outline

- **Part 1:** Set up your environment and understand MLOps principles
- **Part 2:** Analyze wildfire data
- **Part 3:** Deploy analysis to Google Sheets
- **Part 4:** (Optional future work): Deploy a ML model. Monitor model performance and manage updates
- **Deliverable:** Reflection on MLOps deployment

## Learning Objectives
- [ ] Objective 1: Set up AI-enhanced MLOps environment
- [ ] Objective 2: Analyze wildfire data
- [ ] Objective 3: Deploy your analyses to Google Sheets
- [ ] Objective 4: (Optional future work): Deploy a ML model. Monitor model performance and manage updates
- [ ] Objective 5: Reflect on MLOps deployment

## Lab Structure
1. **Setup & Preparation** - Environment setup and MLOps principles
2. **Data Analysis** - Analyzing wildfire data
3. **Deployment to Google Sheets** - Deploying model analyses to Google Sheets
4. **Reflection** - Documenting insights and lessons learned

## Part 1: Environment Setup

Before starting, ensure you have access to the required packages by running the following cell.

These are the libraries needed for data manipulation, analysis, visualization, and interaction with Google Sheets.

### Step 1: Install Required Packages

In [ ]:
# Install required packages
!pip -q install gspread gspread_dataframe

### Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Check if running in Colab
try:
    from google.colab import auth
    import gspread
    from google.auth import default
    from gspread_dataframe import set_with_dataframe
    IN_COLAB = True
except ImportError:
    # Running locally - install packages if needed
    try:
        import gspread
        from google.auth import default
        from gspread_dataframe import set_with_dataframe
        IN_COLAB = False
        print("⚠️  Running locally. Google Sheets authentication requires service account credentials.")
        print("   For Colab, this will work automatically after authentication.")
    except ImportError:
        print("⚠️  gspread not installed. Install with: pip install gspread gspread-dataframe")
        IN_COLAB = False
        gspread = None

### Step 3: Authenticate to Google

Run this cell to authenticate to your Google account. This is necessary to interact with Google Sheets.

In [ ]:
# Authenticate to Google (Colab) or use service account (local)
if IN_COLAB:
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
else:
    # For local execution, you would need to set up service account credentials
    # This is a placeholder - the notebook is designed primarily for Colab
    print("📝 Note: This notebook is optimized for Google Colab.")
    print("   For local execution, set up Google service account credentials.")
    print("   See: https://gspread.readthedocs.io/en/latest/oauth2.html")
    gc = None  # Will need to be configured for local use

## Part 2: Data Analysis

In this part, you will load the wildfire data, perform basic cleaning, and conduct some initial analyses to understand the data.

### Step 4: Load Data

Load the wildfire data from the specified CSV path. If the file is not found, sample data will be created for demonstration. **Ensure the `csv_path` variable in the next cell points to your data file.**

In [ ]:
# Option A: Replace with your actual path (uploaded to Colab files or mounted Drive)
csv_path = "/content/CANADA_WILDFIRES.csv"  # <- change if needed
# Option B: If you uploaded via the left sidebar, it's usually /content/<filename>

# Fallback: if you used this chat upload, adapt path:
# csv_path = "/mnt/data/CANADA_WILDFIRES.csv"

# Try to load the data file (skip if not available during build)
try:
    df_wildfire = pd.read_csv(csv_path)
except FileNotFoundError:
    print(f"⚠️  Data file not found at {csv_path}")
    print("   Creating sample data for demonstration purposes...")
    # Create sample data for build/testing
    import datetime
    dates = pd.date_range(start='2020-01-01', end='2023-12-31', freq='D')
    np.random.seed(42)
    sample_data = {
        'REP_DATE': np.random.choice(dates, size=1000),
        'SIZE_HA': np.random.exponential(scale=100, size=1000),
        'LATITUDE': np.random.uniform(49.0, 70.0, size=1000),
        'LONGITUDE': np.random.uniform(-140.0, -52.0, size=1000),
        'CAUSE': np.random.choice(['H', 'L', 'U', 'H-PB'], size=1000, p=[0.4, 0.4, 0.15, 0.05])
    }
    df_wildfire = pd.DataFrame(sample_data)
    print(f"   Created sample dataset with {len(df_wildfire)} rows")

### Step 5: Basic Cleaning & Standardization

This step standardizes column names to ensure consistency and checks for required columns. Update the `rename_map` if your CSV has different column names.

In [ ]:
# Ensure expected columns exist (rename if your CSV uses different names)
# Expected: REP_DATE, SIZE_HA, LATITUDE, LONGITUDE, CAUSE
rename_map = {
    'rep_date': 'REP_DATE',
    'report_date': 'REP_DATE',
    'size_ha': 'SIZE_HA',
    'lat': 'LATITUDE',
    'lon': 'LONGITUDE',
    'long': 'LONGITUDE',
    'cause_code': 'CAUSE'
}
for old, new in rename_map.items():
    if old in df_wildfire.columns and new not in df_wildfire.columns:
        df_wildfire.rename(columns={old: new}, inplace=True)

required_cols = ['REP_DATE', 'SIZE_HA', 'LATITUDE', 'LONGITUDE', 'CAUSE']
missing = [c for c in required_cols if c not in df_wildfire.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}. "
                     f"Please rename your columns or update the rename_map.")

### Step 6: Perform Analyses

Perform data analysis steps, including converting the date column, extracting the year, analyzing wildfire cause distribution, and calculating the total wildfire size by year.

In [ ]:
# 4.1 Convert 'REP_DATE' to datetime
df_wildfire['REP_DATE'] = pd.to_datetime(df_wildfire['REP_DATE'], errors='coerce')

# 4.2 Extract the year as 'REPORT_YEAR'
df_wildfire['REPORT_YEAR'] = df_wildfire['REP_DATE'].dt.year

# 4.3 Distribution by 'CAUSE' (map to full names where possible)
cause_mapping = {
    'H': 'Human-caused',
    'L': 'Lightning-caused',
    'U': 'Undetermined',
    'H-PB': 'Human-caused (Prescribed Burning)',
    'RE': 'Reclaimed'  # your assumption retained
}
# Use original cause values for items not in mapping
mapped_cause = df_wildfire['CAUSE'].map(cause_mapping).fillna(df_wildfire['CAUSE'])
cause_distribution = mapped_cause.value_counts().sort_values(ascending=False)

print("Wildfire Cause Distribution:\n")
print(cause_distribution)

# 4.4 Trend of total wildfire size (HA) by year
# Guard against NaNs in SIZE_HA
df_wildfire['SIZE_HA'] = pd.to_numeric(df_wildfire['SIZE_HA'], errors='coerce').fillna(0)
yearly_size_trend = df_wildfire.groupby('REPORT_YEAR', dropna=True)['SIZE_HA'].sum().sort_index()
print("\nTotal Wildfire Size (HA) by Year:\n")
print(yearly_size_trend)

### Step 7: Generate Plots

Visualize the analysis results using scatter plots and bar charts to gain insights into wildfire locations, yearly size trends, and cause distribution.

In [ ]:
# Scatter of locations
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df_wildfire, x='LONGITUDE', y='LATITUDE', alpha=0.5, s=5)
plt.title('Wildfire Locations Across Canada')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True)
plt.show()

# Bar of yearly total size
plt.figure(figsize=(15, 7))
yearly_size_trend.plot(kind='bar', color='salmon')
plt.title('Total Wildfire Size (HA) Over Time')
plt.xlabel('Year')
plt.ylabel('Total Size (HA)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Bar of number of wildfires per cause
plt.figure(figsize=(12, 6))
cause_distribution.plot(kind='bar', color='skyblue')
plt.title('Number of Wildfires by Cause')
plt.xlabel('Cause')
plt.ylabel('Number of Wildfires')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Part 3: Deployment to Google Sheets

This part focuses on preparing the analysis results and deploying them to Google Sheets for sharing and further analysis.

### Step 8: Prepare DataFrames for Google Sheets

Prepare the analyzed data into separate DataFrames suitable for pushing to Google Sheets. This includes cause distribution, yearly size trend, and optional metadata.

In [ ]:
df_cause = cause_distribution.rename_axis('Cause').reset_index(name='Count')

df_yearly = yearly_size_trend.rename_axis('Year').reset_index(name='Total_Size_HA')

# Optional metadata sheet: quick snapshot of data health
meta = {
    'Rows': [len(df_wildfire)],
    'Years_Min': [int(df_wildfire['REPORT_YEAR'].min()) if df_wildfire['REPORT_YEAR'].notna().any() else np.nan],
    'Years_Max': [int(df_wildfire['REPORT_YEAR'].max()) if df_wildfire['REPORT_YEAR'].notna().any() else np.nan],
    'Num_Causes': [df_cause.shape[0]],
    'Total_Size_HA': [float(df_wildfire['SIZE_HA'].sum())]
}
df_meta = pd.DataFrame(meta)

### Step 9: Ask for Google Sheet URL and Push Results

Run this cell to push the prepared DataFrames to an empty Google Sheet. You will be prompted to enter the URL of your Google Sheet. **Make sure you have edit access to the sheet.**

In [ ]:
# Only proceed with Google Sheets if gc is available (Colab or properly configured local)
if gc is None:
    print("⚠️  Skipping Google Sheets deployment - not in Colab or credentials not configured")
    print("   In Colab, this section will work after authentication.")
else:
    sheet_url = input("Create an empty google sheet. Paste the Google Sheet URL to write results (must have edit access):\n").strip()

    if not sheet_url:
        raise ValueError("No Google Sheet URL provided.")

    sh = gc.open_by_url(sheet_url)

    def upsert_worksheet(spreadsheet, title, df):
        try:
            ws = spreadsheet.worksheet(title)
            # Clear existing content before writing
            ws.clear()
        except gspread.WorksheetNotFound:
            # Create sheet sized to df
            ws = spreadsheet.add_worksheet(title=title, rows=str(len(df)+10), cols=str(len(df.columns)+10))
        # Write dataframe
        set_with_dataframe(ws, df, include_index=False, include_column_header=True, resize=True)
        return ws

    _ = upsert_worksheet(sh, "CauseDistribution", df_cause)
    _ = upsert_worksheet(sh, "YearlySizeTrend", df_yearly)
    _ = upsert_worksheet(sh, "Metadata", df_meta)

    print("\n✅ Results pushed to Google Sheets:")
    print(" - CauseDistribution")
    print(" - YearlySizeTrend")
    print(" - Metadata")
    print(f"\nSheet: {sheet_url}")

# Show summary even if not pushing to Google Sheets
if gc is None:
    print("\n📊 Local execution - DataFrames created but not pushed to Google Sheets:")
    print(" - CauseDistribution:", df_cause.shape)
    print(" - YearlySizeTrend:", df_yearly.shape)
    print(" - Metadata:", df_meta.shape)

**Congratulations!** You have successfully deployed analysis of real data on a Google sheets! All of these steps above were generated by ChatGPT using prompts similar to this:
- I have a wildfire dataset, I want to do some data nalysis and then deploy the analysis results to a google sheet, and in the interactive notebook in colab, it should ask user to input url to the sheet and then deploy the changes to that url.

Feel free to try it yourself!

## Part 4 (Optional Future Work): Deploy a Prediction Model and Monitor Model Performance

In a real-world MLOps scenario, our work does not stop at data analysis. MLOps also focuses on deploying models for making predictions or decisions.

So, if the implementations above feels easy to follow for you, feel free to actually try deploying a ML prediction model!

The workflow is similar:
- decide on a prediction task (using X to predict Y). For example, using location coordinates to predict wildfire size.
- Select and train a model for this task (we have done this in lab 5-2)
- Deploy the results on Google sheets.

Some prompts you can try:

- "Help me train a model to predict wildfire size based on location and cause."
- "How can I deploy the trained model to make predictions on new data?"
- "Suggest ways to monitor the performance of the wildfire size prediction model."

## Getting Help from Your AI Assistant

**Why AI assistance matters:** AI tools can help you navigate MLOps platforms, suggest deployment strategies, and interpret monitoring results. They're particularly valuable for MLOps where infrastructure knowledge is key.

**Note on Deployment:** In this lab, Google Sheets is used as a straightforward way to demonstrate the concept of deploying analytical results. In real-world MLOps, deployment often involves more complex infrastructure and platforms designed for scalability, automation, and integration with other systems.

**Good prompts for more complex deployment:**
- "Help me choose the right deployment strategy for this model"
- "What monitoring metrics should I track?"
- "How can I optimize this model's performance in production?"
- "What are the trade-offs of this deployment approach?"
- "Help me interpret these monitoring results"
- "How can I automate this MLOps workflow?"

**Avoid vague prompts like "deploy this model"**

**Pro Tip:** Ask "what would an MLOps engineer consider" and "how should I validate this deployment" to get more targeted assistance.

## Part 5: Deliverable: Reflection on MLOps Deployment

Reflect on the MLOps principles applied in this lab. Consider the benefits of this workflow, potential challenges in a production environment, and how AI tools could further enhance each stage of the MLOps lifecycle. Document your thoughts and insights.

Congratulations on completing the lab! You have gained hands-on experience with a basic MLOps workflow, from data analysis to deploying results on Google Sheets.